# Per-trial z distributions after baseline correction (D0121, LFMI5-9)

Picking `max_abs_z` in `make_epoched_data.py` by looking at the data instead of guessing.

Two views per electrode, both in a grid:

1. **Trial traces over the mean** - every trial in light gray under the across-trial mean, the
   same idiom as the electrode traces under the ROI mean in `power/plots.py`, one level down.
2. **Distribution of each trial's largest excursion** - `max |z|` over the epoch per
   (trial, electrode), which is exactly the statistic `max_abs_z` thresholds on, so the
   histogram and the cutoff are in the same units.

Then a survival curve and a cost table: how many trials each candidate threshold removes,
for the problem electrodes and for every channel in the subject.

**Read this before running:** the epochs you load must come from a run with `max_abs_z=None`
(no `_zmax_` in the folder name). If the z-rejection already ran, the trials you are trying to
see are NaN and the distribution you are looking at is the one after the cutoff, not before it.
Cell 3 checks this and warns.


In [ ]:
# === imports & path ===
import os
import sys

try:
    current_script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_script_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.utils.general_utils import load_mne_objects
from src.analysis.vis.trial_z_distribution_vis import (
    per_trial_max_abs_z,
    describe_z_per_channel,
    summarize_z_threshold_tradeoff,
    plot_trial_traces_over_mean_grid,
    plot_z_distribution_grid,
    plot_z_survival,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)


In [ ]:
# === config ===
SUBJECT    = 'D0121'
ELECTRODES = ['LFMI5', 'LFMI6', 'LFMI7', 'LFMI8', 'LFMI9']
TASK       = 'GlobalLocal'
LAB_ROOT   = None       # auto-detect (Box on Windows/mac, /cwork on the cluster)

# Must be a run WITHOUT the z-rejection, i.e. no `_zmax_` suffix. Swap for
# whichever run you actually have on disk.
EPOCHS_ROOT = ('Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_'
               'outliers_10_drop_and_nan_thresh_perc_5.0_70.0-150.0_Hz_padLength_3.0s_'
               'filterbank_hilbert_stat_func_ttest_ind_equal_var_False_nan_policy_omit')

# 'HG_ev1_power_rescaled' is the object make_epoched_data.py builds the rejection
# mask from, so it is the one to threshold on. 'HG_ev1_rescaled' is the amplitude
# view of the same trials -- worth a second pass through this notebook, since a
# 30 z power excursion is only ~5 z in amplitude.
MNE_OBJ_TYPE = 'HG_ev1_power_rescaled'

ACC_TRIALS_ONLY = False   # False here on purpose: an artifact does not care
                          # whether the response was correct, and you want to
                          # see every trial the threshold would act on.

# Candidate cutoffs to draw and cost out. 30 is the current default in main().
THRESHOLDS = [5, 10, 15, 20, 30, 50, 100]

FIG_DIR = Path(project_root) / 'src' / 'analysis' / 'vis' / 'figs' / 'trial_z_distributions' / SUBJECT
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('figures ->', FIG_DIR)


In [ ]:
# === load, and check what has already been removed ===
if '_zmax_' in EPOCHS_ROOT:
    print('WARNING: this run already applied max_abs_z. The extreme trials are '
          'already NaN, so the distribution below is post-rejection and cannot '
          'be used to choose the threshold. Re-run make_epoched_data with '
          'max_abs_z=None to see the raw distribution.\n')

mne_objects = load_mne_objects(SUBJECT, EPOCHS_ROOT, TASK,
                               just_HG_ev1_rescaled=True, LAB_root=LAB_ROOT)
epochs = mne_objects[MNE_OBJ_TYPE]
if ACC_TRIALS_ONLY:
    epochs = epochs['Accuracy1.0']

print(f'{SUBJECT}: {len(epochs)} trials, {len(epochs.ch_names)} channels, '
      f'sfreq={epochs.info["sfreq"]:.1f}, tmin={epochs.tmin}, tmax={epochs.tmax}')

present = [ch for ch in ELECTRODES if ch in epochs.ch_names]
missing = [ch for ch in ELECTRODES if ch not in epochs.ch_names]
print('present:', present)
if missing:
    print('missing (dropped upstream by drop_and_impute or --electrodes-to-drop):', missing)

# How much of the data the *first* (raw-voltage) outlier pass already took out.
data = epochs.get_data()
pair_nan = np.all(np.isnan(data), axis=2)
print(f'\nalready fully NaN: {pair_nan.sum()} / {pair_nan.size} (trial, channel) pairs '
      f'({100 * pair_nan.mean():.2f}%) across {int(pair_nan.any(axis=1).sum())} trials')
del data


## 1. Trial traces over the mean

Two passes over the same panels. The first is the full y range: with a 20,000 z trial in the
set every other trace collapses onto zero, which is itself the finding. The second is `symlog`,
which keeps the bulk readable and the artifacts visible at the same time.


In [ ]:
# Full y range -- shows how far the worst trials actually go.
fig = plot_trial_traces_over_mean_grid(
    epochs, electrodes=present,
    thresholds=[30],
    title=f'{SUBJECT} {MNE_OBJ_TYPE}: trials (gray) over across-trial mean (red), full range',
    grid_shape=(2, 3),
    save_path=FIG_DIR / f'{SUBJECT}_trial_traces_full_range.png',
)
plt.show()


In [ ]:
# symlog: linear within +/-1 z, log beyond. The bulk and the tail in one panel.
fig = plot_trial_traces_over_mean_grid(
    epochs, electrodes=present,
    thresholds=THRESHOLDS,
    yscale='symlog',
    title=f'{SUBJECT} {MNE_OBJ_TYPE}: trials over mean, symlog y (dashed = candidate thresholds)',
    grid_shape=(2, 3),
    save_path=FIG_DIR / f'{SUBJECT}_trial_traces_symlog.png',
)
plt.show()

# And the bulk on its own: clip each panel to its own 99.5th percentile of |z|.
fig = plot_trial_traces_over_mean_grid(
    epochs, electrodes=present,
    robust_ylim_pct=99.5,
    title=f'{SUBJECT} {MNE_OBJ_TYPE}: trials over mean, clipped to the 99.5th pctile of |z|',
    grid_shape=(2, 3),
    save_path=FIG_DIR / f'{SUBJECT}_trial_traces_robust.png',
)
plt.show()


## 2. Distribution of the per-trial excursion

`per_trial_max_abs_z` returns, for each (trial, electrode), the largest `|z|` anywhere in the
epoch. That is the same reduction `make_epoched_data.py` applies before comparing to
`max_abs_z`, so a number read off these panels transfers directly.


In [ ]:
stat_lfmi, ch_lfmi, _ = per_trial_max_abs_z(epochs, electrodes=present)
stat_all,  ch_all,  _ = per_trial_max_abs_z(epochs, electrodes=None)

print('per-channel quantiles of the per-trial max |z| -- problem electrodes\n')
display(describe_z_per_channel(stat_lfmi, ch_lfmi).round(2))

print('\nsame, whole subject (10 most extreme channels by max)\n')
all_desc = describe_z_per_channel(stat_all, ch_all)
display(all_desc.sort_values('p100', ascending=False).head(10).round(2))


In [ ]:
fig = plot_z_distribution_grid(
    stat_lfmi, ch_lfmi,
    thresholds=[10, 30, 100],
    title=f'{SUBJECT}: per-trial max |z| ({MNE_OBJ_TYPE}), log10 bins',
    grid_shape=(2, 3),
    save_path=FIG_DIR / f'{SUBJECT}_z_histograms.png',
)
plt.show()


In [ ]:
# Survival curve. The problem electrodes against every channel in the subject:
# the threshold you want is where the curve stops falling smoothly and the last
# points sit detached from the body of the distribution.
fig = plot_z_survival(
    {'LFMI5-9': stat_lfmi, f'all {len(ch_all)} channels': stat_all},
    thresholds=THRESHOLDS,
    title=f'{SUBJECT}: fraction of (trial, channel) pairs above a given max |z|',
    save_path=FIG_DIR / f'{SUBJECT}_z_survival.png',
)
plt.show()


## 3. What each threshold would cost

The threshold has to be defensible on the whole subject, not just on the electrodes you already
suspect, so cost it out on both. Pick the smallest value that clears the detached tail without
biting into the body of the distribution -- concretely, one where `pct_of_scorable_pairs` on the
all-channel table is still a small fraction of a percent.


In [ ]:
print('cost on the problem electrodes\n')
display(summarize_z_threshold_tradeoff(stat_lfmi, ch_lfmi, THRESHOLDS).round(3))

print('\ncost on every channel in the subject\n')
display(summarize_z_threshold_tradeoff(stat_all, ch_all, THRESHOLDS).round(3))


In [ ]:
# The trials a given threshold would remove, by name, so you can go look at them.
THR = 30
over = np.zeros_like(stat_lfmi, dtype=bool)
np.greater(stat_lfmi, THR, out=over, where=~np.isnan(stat_lfmi))
rows = [{'trial': int(t), 'channel': ch_lfmi[c], 'max_abs_z': stat_lfmi[t, c]}
        for t, c in zip(*np.nonzero(over))]
rejected = pd.DataFrame(rows).sort_values('max_abs_z', ascending=False)
print(f'{len(rejected)} (trial, channel) pairs over |z| > {THR} on {present}')
display(rejected.head(30))

if epochs.metadata is not None and len(rejected):
    cols = [c for c in ['congruency', 'incongruent_proportion', 'task_sequence',
                        'switch_proportion', 'accuracy', 'trial_count']
            if c in epochs.metadata.columns]
    if cols:
        print('\nmetadata for the affected trials -- if these cluster in one block type, '
              'the rejection is not condition-neutral and you have to say so:')
        display(epochs.metadata.iloc[sorted(set(rejected['trial']))][cols])


## 4. Same thing on the amplitude view (optional)

Set `MNE_OBJ_TYPE = 'HG_ev1_rescaled'` in the config cell and re-run. Power is amplitude
squared, so a threshold of 30 on power is about 5.5 on amplitude -- worth confirming the two
views agree about which trials are bad before committing to a number.


## 5. Hard z threshold vs. `outliers_to_nan`, and why the pipeline uses both

Short version: they are not alternatives, and the reason the current code runs both is that each
one fails in a way the other catches.

**What `outliers_to_nan` does.** It is an N-SD rule computed *per channel across trials*, run on
raw voltage before `gamma.extract`. Two consequences:

- *It is relative, so it always finds something.* At 10 SD on a clean channel it removes the
  cleanest data you have; on a channel with one 300x excursion the SD is itself inflated by that
  excursion, so the rule set to catch it is calibrated by it. That is exactly how a 278 z trial
  survived the first pass (`make_epoched_data.py:305-317`).
- *It runs on voltage, before squaring.* Power ratio is amplitude ratio squared, so an excursion
  that is unremarkable in voltage on one contact is enormous in rescaled power. It is also
  per channel, so it NaNs the trial on whichever contact the excursion is largest and leaves the
  neighbour untouched: D0121 trial 367 was NaN'd on LFMI8/9 and left at 331 z on LFMI5.

**What the absolute threshold does.** After `rescale(..., mode='zscore')` the units are already
normalized per channel against that channel's own baseline, so z is comparable across trials,
channels, and subjects. An absolute cutoff in those units means the same thing everywhere, does
not get dragged upward by the artifacts it is meant to remove, and applies one mask across all
four derived objects so the amplitude and power views agree on which trials exist.

**So: keep both.** Run `outliers_to_nan` (`outlier_policy='drop_and_nan'`, `outliers=10`) as the
first pass on voltage, and a `max_abs_z` pass after baseline correction as the backstop for what
squaring reveals. Dropping the hard threshold in favour of `outliers_to_nan` alone puts you back
in the situation this analysis exists to fix. Dropping `outliers_to_nan` in favour of the
threshold alone is also wrong: the voltage pass runs before the Hilbert filter, so it stops a
transient from smearing across the filter's impulse response into neighbouring timepoints, which
no post-hoc threshold can undo.

**Three things to be careful about, whichever you use:**

1. *Both mark timepoints, not trials.* `outliers_to_nan` NaNs the offending samples; the
   `max_abs_z` pass NaNs the whole (trial, channel) trace. Everything downstream has to be
   NaN-aware, which is why the evoked builders in `general_utils.py` use `np.nanmean`/`nanstd`
   with per-sample valid counts. If you add a step that uses plain `mean`, it will silently
   produce NaN traces.
2. *Rejection is not automatically condition-neutral.* Removing the most extreme trials removes
   variance, and if the removed trials cluster in one block type, condition and rejection are
   confounded and it lands on the interaction term. The last code cell above checks this for the
   threshold you pick -- run it.
3. *`drop_and_impute` is the one option to think hardest about.* Imputing an outlier trial with
   the channel mean does not just remove an artifact, it inserts a trial that is by construction
   average. That shrinks the within-channel variance the ANOVA divides by and inflates F. If you
   are running the windowed ANOVA, prefer `drop_and_nan` and let the NaN-aware code drop it.

**On picking the number.** From the survival curve, take the smallest cutoff that sits in the gap
between the body of the distribution and the detached points. Given the ~24,000 z excursions
described in `make_epoched_data.py`, anything between 20 and 50 will catch them; the current
default of 30 is a reasonable place to land if the tables above show it costing well under a
percent of pairs across all channels. What matters more than the exact value is that the result
is insensitive to it: re-run the power traces at two or three thresholds in that range, and if
the effect moves, the effect was the artifact.
